# RE_101 · On i quan hi ha accidents a Barcelona

**Optativa d'Aprenentatge automàtic · Bloc 1 — Dades · Dificultat 1**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Ets analista de dades a l'àrea de Mobilitat de l'Ajuntament de Barcelona. La regidoria
vol decidir on posar més controls i radars pedagògics l'any que ve, i et passa tots els accidents
que va gestionar la Guàrdia Urbana el 2024.

**La pregunta:** en quins districtes, a quines hores i quins dies hi ha més accidents, i on fan
més mal?

L'enunciat complet, amb el diccionari de columnes, és a [RE_101](../RE_101_accidents_bcn.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FITXER = "accidents_gu_bcn_2024.csv"
URL = "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/Machine%20Learning/04_reptes/dades/" + FITXER
ruta = URL
for candidata in (os.path.join("dades", FITXER), os.path.join("..", "dades", FITXER)):
    if os.path.exists(candidata):
        ruta = candidata
df = pd.read_csv(ruta)
print(ruta, df.shape)

## Nivell 1 · Explorar

### 1.1 · Forma i tipus

Quantes files i quantes columnes té la taula? Quantes columnes són de text i quantes numèriques?

In [ ]:
print(df.shape)
print(df.dtypes.value_counts())

### 1.2 · Inventari de nuls

Fes una taula amb el nombre de buits i el percentatge de buits de cada columna, ordenada de més buida a menys. Deixa només les columnes que en tenen.

In [ ]:
buits = df.isna().sum()
nuls = pd.DataFrame({"buits": buits, "pct": (100 * buits / len(df)).round(1)})
nuls = nuls[nuls["buits"] > 0].sort_values("buits", ascending=False)
nuls

### 1.3 · Què és rar

Mira els noms de les columnes un per un, i els valors de `Nom_districte`, `Descripcio_torn` i `Num_postal`. Apunta en una cel·la de text **dues coses que et sorprenguin** o que et facin desconfiar.

In [ ]:
print(df.columns.tolist())
print(df["Nom_districte"].value_counts())
print(df["Descripcio_torn"].value_counts())
print(df["Num_postal "].value_counts().head(10))
print(df[["Numero_morts", "Numero_lesionats_greus", "Numero_victimes"]].describe())

**Comentari.** El que hi ha: el nom `Num_postal ` acaba amb un espai; `Num_postal` barreja números, rangs (`423-425`) i punts quilomètrics (`K165`); `Numero_morts`, `Numero_lesionats_greus` i `Numero_victimes` no tenen mai el valor 0, i el mínim és 1. És a dir, **els NaN volen dir 0**. Al 2023 hi havia zeros explícits (ho veuràs al RE_302).

## Nivell 2 · Respondre

### 2.1 · Accidents per districte

Quants accidents hi ha a cada districte? Ordena'ls de més a menys i fes-ne un gràfic de barres.

In [ ]:
per_districte = df.groupby("Nom_districte").size().sort_values(ascending=False)
print(per_districte)
print("suma:", per_districte.sum(), "files:", len(df))
per_districte.plot(kind="bar", title="Accidents per districte (2024)")
plt.ylabel("accidents")
plt.show()

### 2.2 · On fan més mal

Per a cada districte, calcula el **total de víctimes** i les **víctimes per accident**. Abans, decideix què fas amb els buits de `Numero_victimes` i escriu en una línia per què. Quin districte té més víctimes per accident? És el mateix que el de 2.1?

In [ ]:
df["victimes"] = df["Numero_victimes"].fillna(0)
mal = df.groupby("Nom_districte")["victimes"].agg(["sum", "mean"]).sort_values("mean", ascending=False)
print(mal.round(3))
print("total taula:", mal["sum"].sum(), "· total df:", df["Numero_victimes"].sum())
# el mateix sense omplir els buits: la mitjana es fa només sobre els accidents amb víctimes
print(df.groupby("Nom_districte")["Numero_victimes"].mean().round(3).sort_values(ascending=False))

**Comentari.** El parany és fer la mitjana sense omplir: `mean()` ignora els NaN i la mitjana surt calculada només sobre accidents amb víctimes (sempre ≥ 1). Les diferències entre districtes són petites: el que separa els districtes és el nombre d'accidents, no la gravetat.

### 2.3 · Hora del dia

Quants accidents hi ha a cada hora del dia? Fes-ne un gràfic **ordenat per hora** (de 0 a 23), no per freqüència. A quines hores hi ha el pic?

In [ ]:
per_hora = df.groupby("Hora_dia").size()
print(len(per_hora), per_hora.sum())
print(per_hora.sort_values(ascending=False).head(5))
per_hora.plot(kind="bar", title="Accidents per hora del dia")
plt.show()

### 2.4 · Dia de la setmana

Quants accidents hi ha cada dia de la setmana? Mostra'ls en l'ordre natural, **de dilluns a diumenge**, no per ordre alfabètic, i fes-ne un gràfic.

In [ ]:
dies = ["Dilluns", "Dimarts", "Dimecres", "Dijous", "Divendres", "Dissabte", "Diumenge"]
per_dia = df["Descripcio_dia_setmana"].value_counts().reindex(dies)
print(per_dia, per_dia.sum())
per_dia.plot(kind="bar", title="Accidents per dia de la setmana")
plt.show()

### 2.5 · Lluny o a prop del centre (NumPy)

Calcula, **amb NumPy**, la distància en metres de cada accident a la plaça de Catalunya
(latitud 41,3870, longitud 2,1700). Per passar graus a metres a Barcelona fes servir aquesta
aproximació:

- 1 grau de latitud ≈ 111.000 m;
- 1 grau de longitud ≈ 111.000 · cos(latitud) m.

Construeix un array `D` de forma `(n, 2)` amb les diferències en metres (nord-sud i est-oest), calcula
la norma de cada fila i desa-la com a columna `dist_m`. Quants accidents hi ha a menys d'1 km de la
plaça? De quins barris són els 10 més llunyans?

In [ ]:
coords = df[["Latitud_WGS84", "Longitud_WGS84"]].to_numpy()
escala = np.array([111000, 111000 * np.cos(np.radians(41.387))])
D = (coords - np.array([41.387, 2.170])) * escala
print(D.shape)
df["dist_m"] = np.linalg.norm(D, axis=1)
print(df["dist_m"].describe().round(0))
print("a menys d'1 km:", (df["dist_m"] < 1000).sum())
df.nlargest(10, "dist_m")[["Nom_districte", "Nom_barri", "Nom_carrer", "dist_m"]]

## Nivell 3 · Més enllà

### 3.1 · Les altres coordenades

El fitxer també porta les coordenades **UTM** (`Coordenada_UTM_X_ED50` i
`Coordenada_UTM_Y_ED50`), que ja són metres: la X és l'est i la Y és el nord. A la plaça de
Catalunya valen, aproximadament, X = 430.700 m i Y = 4.582.300 m. Torna a calcular la distància de
cada accident a la plaça amb aquestes columnes i compara-la amb la de 2.5. Han de coincidir? Si no
coincideixen, troba per què.

In [ ]:
print(df[["Coordenada_UTM_X_ED50", "Coordenada_UTM_Y_ED50"]].describe().round(0))
mal_fet = np.linalg.norm(df[["Coordenada_UTM_X_ED50", "Coordenada_UTM_Y_ED50"]].to_numpy() - [430700, 4582300], axis=1)
ben_fet = np.linalg.norm(df[["Coordenada_UTM_Y_ED50", "Coordenada_UTM_X_ED50"]].to_numpy() - [430700, 4582300], axis=1)
print("amb les columnes tal com es diuen:", np.median(mal_fet).round(0))
print("intercanviant-les:", np.median(ben_fet).round(0))
print("diferència mediana amb 2.5 (m):", np.median(np.abs(ben_fet - df["dist_m"])).round(1))

**Comentari.** Les columnes UTM estan **intercanviades**: la que es diu X té valors de 4,58 milions (és el nord). Intercanviant-les, la distància coincideix amb la de 2.5 amb diferències de l'ordre de desenes de metres (ED50 contra WGS84 i l'aproximació).

### 3.2 · Torn i dia

Fes una taula amb els torns (`Descripcio_torn`) a les files i els dies de la setmana a les columnes, amb el nombre d'accidents a cada casella. Els accidents de nit es concentren en algun dia?

In [ ]:
taula = df.groupby(["Descripcio_torn", "Descripcio_dia_setmana"]).size().unstack()[dies]
print(taula, taula.to_numpy().sum())

## Conclusió de referència

L'Eixample concentra 1.898 dels 7.536 accidents del 2024 (un 25 %), gairebé el doble que Sant
Martí (948), però és on hi ha més trànsit: per víctimes per accident (comptant els buits com a 0)
els districtes queden molt a prop, entre 1,02 (Ciutat Vella) i 1,23 (Les Corts). Els accidents es
concentren entre les 13 h i les 18 h (pic a les 13 h, 523) i en dies laborables (uns 1.200 cada
dia contra 674 en diumenge). 692 accidents van passar a menys d'1 km de la plaça de Catalunya; els
més llunyans són a Vallbona i Ciutat Meridiana, a uns 9 km. Paranys: els NaN de víctimes volen dir
0, i les columnes UTM X i Y estan intercanviades.